# Script 2 — Preparação & Engenharia de Features (V3 — DFP + ITR)
**TCC: Predição de Indicadores Financeiros Corporativos com ML e IA Generativa**

Esta versão incorpora **DFPs e ITRs** como observações de treino, aumentando o dataset
de ~74 para ~369 observações — um ganho de 5× sem alterar o Script 1.

## Estratégia de target: "próximo DFP anual"

Cada observação (seja DFP ou ITR) prediz o **próximo DFP anual estritamente posterior**
à sua data de referência:

| Observação | Prediz |
|---|---|
| ITR Q1/2022 (mar) | DFP 2022 (dez) |
| ITR Q2/2022 (jun) | DFP 2022 (dez) |
| ITR Q3/2022 (set) | DFP 2022 (dez) |
| DFP 2022 (dez)    | DFP 2023 (dez) |

Isso preserva a interpretação econômica — o modelo aprende a relação entre sinais
intermediários e o resultado anual fechado — sem misturar periodicidades nos targets.

## Tratamento de autocorrelação

Observações do mesmo empresa no mesmo ciclo anual são correlacionadas por construção.
O split temporal e o GroupKFold por empresa garantem que o modelo de validação nunca
vê observações da mesma empresa em treino e validação simultaneamente.

## Etapa 0 — Dependências, logging e configuração global

In [1]:
import logging
import json
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_selection import RFE
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.max_columns', 40)

# ── Logging estruturado ───────────────────────────────────────────────────
logger = logging.getLogger('pipeline_preparacao_v3')
logger.setLevel(logging.DEBUG)
logger.handlers.clear()

PASTA_SAIDA = Path('outputs')
PASTA_SAIDA.mkdir(exist_ok=True)

_fmt = logging.Formatter('%(asctime)s | %(levelname)-8s | %(message)s',
                         datefmt='%Y-%m-%d %H:%M:%S')
_sh = logging.StreamHandler()
_sh.setLevel(logging.INFO)
_sh.setFormatter(_fmt)
logger.addHandler(_sh)

_fh = logging.FileHandler(PASTA_SAIDA / 'pipeline_preparacao_v3.log',
                          mode='w', encoding='utf-8')
_fh.setLevel(logging.DEBUG)
_fh.setFormatter(_fmt)
logger.addHandler(_fh)

# ── Parâmetros configuráveis ──────────────────────────────────────────────
LISTA_KPIS = [
    'margem_bruta', 'margem_ebit', 'margem_liquida', 'margem_ebitda',
    'roe', 'roa', 'liquidez_corrente', 'liquidez_imediata',
    'endividamento', 'alavancagem_de', 'div_liquida', 'cobertura_juros',
    'giro_ativo', 'fco_receita', 'fco_lucro', 'EBITDA',
    'FCF', 'margem_fcf', 'conversao_caixa',
]

TARGET_COLS = {
    'TARGET_DRE_3.01': 'DRE_3.01',
    'TARGET_DRE_3.11': 'DRE_3.11',
    'TARGET_EBITDA'  : 'EBITDA',
}

LIMIAR_NULO    = 0.80
FATOR_WINSOR   = 3.0
MAX_COLS_YOY   = 16
CLIP_YOY       = 5.0
CORR_MIN       = 0.10
N_FEATURES_RFE = 15
FRAC_TREINO    = 0.75
# Tolerância em dias para considerar gap YoY válido (~1 ano ± 25 dias)
GAP_YOY_MIN    = 340
GAP_YOY_MAX    = 395

logger.info("Script 2 V3 (DFP+ITR) iniciado | kpis=%d | limiar_nulo=%.0f%% | "
            "winsor=%.1f | yoy_max=%d | rfe_n=%d | frac_treino=%.0f%%",
            len(LISTA_KPIS), LIMIAR_NULO*100, FATOR_WINSOR,
            MAX_COLS_YOY, N_FEATURES_RFE, FRAC_TREINO*100)


2026-04-22 12:11:19 | INFO     | Script 2 V3 (DFP+ITR) iniciado | kpis=19 | limiar_nulo=80% | winsor=3.0 | yoy_max=16 | rfe_n=15 | frac_treino=75%


## Etapa 1 — Carregamento e validação do dataset consolidado

Lê o Parquet do Script 1 e executa verificações de integridade.
O dataset contém **471 linhas** (99 DFP + 372 ITR) × 736 colunas.

In [2]:
cam_parquet = PASTA_SAIDA / 'dataset_cvm_consolidado.parquet'
if not cam_parquet.exists():
    raise FileNotFoundError(
        f"Parquet não encontrado: {cam_parquet}\n"
        "Execute o Script 1 (01_cvm_processamento_V5.ipynb) antes de continuar."
    )

dataset = pd.read_parquet(cam_parquet)
logger.info("Dataset carregado: %d × %d", *dataset.shape)

# ── Normalização de tipos ─────────────────────────────────────────────────
# DT_REFER: preserva timezone America/Sao_Paulo gravado pelo Script 1
dataset['DT_REFER'] = pd.to_datetime(dataset['DT_REFER'], errors='coerce', utc=False)
dataset['ANO']      = dataset['DT_REFER'].dt.year.astype('Int64')
dataset['TRIMESTRE']= dataset['DT_REFER'].dt.quarter.astype('Int64')
dataset['MES']      = dataset['DT_REFER'].dt.month.astype('Int64')

# ── Validações obrigatórias ───────────────────────────────────────────────
COLS_OBR = ['CNPJ_CIA', 'NOME_CIA', 'SETOR', 'ANO', 'ORIGEM', 'DT_REFER']
faltando = [c for c in COLS_OBR if c not in dataset.columns]
if faltando:
    raise ValueError(f"Colunas obrigatórias ausentes: {faltando}")

n_emp = dataset['NOME_CIA'].nunique()
if n_emp < 25:
    logger.warning("Apenas %d/25 empresas no dataset", n_emp)
else:
    logger.info("Empresas: %d/25 ✅", n_emp)

kpis_presentes = [k for k in LISTA_KPIS if k in dataset.columns]
kpis_ausentes  = [k for k in LISTA_KPIS if k not in dataset.columns]
if kpis_ausentes:
    logger.warning("KPIs ausentes: %s", kpis_ausentes)

# Cobertura por origem
for orig in ['DFP', 'ITR']:
    sub = dataset[dataset['ORIGEM'] == orig]
    logger.info("%-3s: %d obs | %d empresas | anos %s",
                orig, len(sub), sub['NOME_CIA'].nunique(),
                sorted(sub['ANO'].dropna().astype(int).unique()))

print(f"\n{'='*60}")
print(f"  Dataset carregado")
print(f"  Shape    : {dataset.shape[0]:,} × {dataset.shape[1]}")
print(f"  Empresas : {n_emp} / 25")
print(f"  DFP      : {(dataset['ORIGEM']=='DFP').sum()} obs")
print(f"  ITR      : {(dataset['ORIGEM']=='ITR').sum()} obs")
print(f"  KPIs     : {len(kpis_presentes)} / {len(LISTA_KPIS)}")
print(f"{'='*60}")


2026-04-22 12:11:19 | INFO     | Dataset carregado: 471 × 736
2026-04-22 12:11:19 | INFO     | Empresas: 25/25 ✅
2026-04-22 12:11:19 | INFO     | DFP: 99 obs | 25 empresas | anos [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
2026-04-22 12:11:19 | INFO     | ITR: 372 obs | 25 empresas | anos [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]



  Dataset carregado
  Shape    : 471 × 736
  Empresas : 25 / 25
  DFP      : 99 obs
  ITR      : 372 obs
  KPIs     : 19 / 19


## Etapa 2 — Deduplicação intra-período

Garante exatamente uma linha por `(CNPJ_CIA, DT_REFER, ORIGEM)`.
Retificações da CVM (versões múltiplas do mesmo documento) são resolvidas
mantendo o registro com maior completude de KPIs — proxy da versão mais recente.

In [3]:
# ── Deduplicação por (empresa, data, origem) ─────────────────────────────
n_antes = len(dataset)
dataset['_n_kpis'] = dataset[kpis_presentes].notna().sum(axis=1)

dataset = (dataset
    .sort_values(['CNPJ_CIA', 'DT_REFER', 'ORIGEM', '_n_kpis'],
                 ascending=[True, True, True, False])
    .drop_duplicates(subset=['CNPJ_CIA', 'DT_REFER', 'ORIGEM'], keep='first')
    .drop(columns=['_n_kpis'])
    .sort_values(['CNPJ_CIA', 'DT_REFER'])
    .reset_index(drop=True)
)
rem = n_antes - len(dataset)
logger.info("Deduplicação: %d → %d (-%d retificações)", n_antes, len(dataset), rem)

# Verificar: Arezzo tem DFP e ITR na mesma DT_REFER (31/dez)?
# Isso é esperado — são documentos diferentes (ORIGEM distinta), ambos válidos
dtref_dup = dataset.groupby(['CNPJ_CIA','DT_REFER']).size()
multi = dtref_dup[dtref_dup > 1]
if not multi.empty:
    logger.info("Datas com DFP e ITR simultâneos (esperado para Q4): %d casos", len(multi))
    logger.debug("Detalhes:\n%s", multi.reset_index().head(5).to_string())

print(f"Dataset após dedup: {dataset.shape} | removidas: {rem}")


2026-04-22 12:11:19 | INFO     | Deduplicação: 471 → 471 (-0 retificações)
2026-04-22 12:11:19 | INFO     | Datas com DFP e ITR simultâneos (esperado para Q4): 1 casos


Dataset após dedup: (471, 736) | removidas: 0


## Etapa 3 — Remoção de colunas com excesso de nulos

Remove colunas com `>80%` de nulos **calculado sobre o dataset completo** (DFP+ITR).
Os 19 KPIs são protegidos independentemente da taxa de nulos.

In [4]:
COLS_PROTEGIDAS = set(kpis_presentes + ['ANO', 'TRIMESTRE', 'MES'])
cols_num   = dataset.select_dtypes(include='number').columns.tolist()
cols_cand  = [c for c in cols_num if c not in COLS_PROTEGIDAS]

taxa_nulo  = dataset[cols_cand].isnull().mean()
cols_excluir = taxa_nulo[taxa_nulo > LIMIAR_NULO].index.tolist()

grupos_exc = {}
for c in cols_excluir:
    p = c.split('_')[0]
    grupos_exc[p] = grupos_exc.get(p, 0) + 1
logger.info("Remoção >%.0f%% nulos: %d colunas | grupos: %s",
            LIMIAR_NULO*100, len(cols_excluir),
            dict(sorted(grupos_exc.items(), key=lambda x: -x[1])))

dataset = dataset.drop(columns=cols_excluir)
kpis_presentes = [k for k in kpis_presentes if k in dataset.columns]

print(f"Removidas: {len(cols_excluir)} colunas (>{LIMIAR_NULO:.0%} nulos)")
print(f"Dataset : {dataset.shape[0]} × {dataset.shape[1]}")
print(f"KPIs    : {len(kpis_presentes)} / {len(LISTA_KPIS)}")


2026-04-22 12:11:19 | INFO     | Remoção >80% nulos: 289 colunas | grupos: {'BPP': 78, 'BPA': 66, 'DRE': 38, 'DVA': 35, 'DMPL': 32, 'DFC': 30, 'DRA': 10}


Removidas: 289 colunas (>80% nulos)
Dataset : 471 × 447
KPIs    : 19 / 19


## Etapa 4 — Imputação por mediana do setor

Imputa nulos nos KPIs usando mediana do setor como nível primário e
mediana global como fallback. A imputação é calculada separadamente
por `(SETOR, ORIGEM)` — a mediana de um ITR do setor Petróleo é a referência
correta para um ITR faltante desse setor, não a mediana de DFPs.

In [5]:
n_nulos_pre = dataset[kpis_presentes].isnull().sum().sum()

for kpi in kpis_presentes:
    n = dataset[kpi].isnull().sum()
    if n == 0:
        continue
    # Mediana por (setor, origem) — respeita diferença entre DFP e ITR
    med_setor_origem = dataset.groupby(['SETOR', 'ORIGEM'])[kpi].transform('median')
    # Fallback 1: mediana por setor (ignora origem)
    med_setor        = dataset.groupby('SETOR')[kpi].transform('median')
    # Fallback 2: mediana global
    med_global       = dataset[kpi].median()

    dataset[kpi] = (dataset[kpi]
        .fillna(med_setor_origem)
        .fillna(med_setor)
        .fillna(med_global)
    )
    n_rest = dataset[kpi].isnull().sum()
    if n_rest:
        logger.warning("Imputação incompleta: %s | %d nulos restantes", kpi, n_rest)

n_nulos_pos = dataset[kpis_presentes].isnull().sum().sum()
logger.info("Imputação: %d → %d nulos nos KPIs", n_nulos_pre, n_nulos_pos)
print(f"Nulos KPIs: {n_nulos_pre} → {n_nulos_pos}")


2026-04-22 12:11:19 | INFO     | Imputação: 42 → 0 nulos nos KPIs


Nulos KPIs: 42 → 0


## Etapa 5 — Winsorização de outliers por setor

Trunca valores extremos de cada KPI dentro de cada setor usando `Q1 − 3×IQR`
e `Q3 + 3×IQR`. Implementado com `groupby().transform()` para preservar a
coluna SETOR (compatível com pandas >= 2.0).

In [6]:
def winsorizacao_setor(df_in: pd.DataFrame, col: str, fator: float = 3.0) -> pd.DataFrame:
    """
    Winsorização por setor via groupby+transform.
    transform() preserva a estrutura do DataFrame — SETOR nunca é dropado.
    """
    def _clip(serie: pd.Series) -> pd.Series:
        q1  = serie.quantile(0.25)
        q3  = serie.quantile(0.75)
        iqr = q3 - q1
        if iqr == 0:
            return serie
        return serie.clip(lower=q1 - fator * iqr, upper=q3 + fator * iqr)

    df_out = df_in.copy()
    df_out[col] = df_in.groupby('SETOR')[col].transform(_clip)
    return df_out

n_clip_total = 0
for kpi in kpis_presentes:
    antes = dataset[kpi].copy()
    dataset = winsorizacao_setor(dataset, kpi, FATOR_WINSOR)
    n_clip_total += (dataset[kpi] != antes).sum()

assert 'SETOR' in dataset.columns, "SETOR perdido na winsorização"
logger.info("Winsorização: fator=%.1f | %d valores truncados", FATOR_WINSOR, n_clip_total)
print(f"✅ Winsorização | fator={FATOR_WINSOR} | {n_clip_total} valores truncados")


2026-04-22 12:11:20 | INFO     | Winsorização: fator=3.0 | 241 valores truncados


✅ Winsorização | fator=3.0 | 241 valores truncados


## Etapa 6 — Engenharia de features YoY (mesmo trimestre, ano anterior)

Com dados trimestrais misturados, o YoY correto é a variação em relação ao
**mesmo trimestre do ano anterior** — não ao período imediatamente anterior.

Técnica: `shift(4)` dentro de cada empresa ordena 4 posições para trás na sequência
temporal. Se a empresa tem Q1/Q2/Q3/DFP por ano, o shift(4) de Q1/2023 aponta para
Q1/2022 — exatamente 1 ano antes. O gap entre as datas é validado para garantir
que está entre 340 e 395 dias. Gaps fora desse intervalo (empresa com dados incompletos
em algum ano) são anulados.

In [7]:
dataset = dataset.sort_values(['CNPJ_CIA', 'DT_REFER']).reset_index(drop=True)

# KPIs usados para YoY (exclui métricas absolutas)
KPIS_YOY = [k for k in kpis_presentes
             if k not in ('div_liquida', 'EBITDA', 'FCF')][:MAX_COLS_YOY]

cols_yoy = []

# Pré-calcular gap temporal para validação de continuidade
dataset['_dt_prev'] = dataset.groupby('CNPJ_CIA')['DT_REFER'].shift(4)
dataset['_gap_dias'] = (dataset['DT_REFER'] - dataset['_dt_prev']).dt.days
gap_invalido = ~dataset['_gap_dias'].between(GAP_YOY_MIN, GAP_YOY_MAX)

n_gap_invalidos = gap_invalido.sum()
logger.info("YoY: gaps inválidos anulados: %d / %d obs (%.1f%%)",
            n_gap_invalidos, len(dataset), n_gap_invalidos/len(dataset)*100)

for kpi in KPIS_YOY:
    col_yoy = f'{kpi}_yoy'
    prev    = dataset.groupby('CNPJ_CIA')[kpi].shift(4)
    yoy_raw = ((dataset[kpi] - prev) / prev.abs().replace(0, np.nan))
    yoy_raw = yoy_raw.replace([np.inf, -np.inf], np.nan).clip(-CLIP_YOY, CLIP_YOY)
    yoy_raw[gap_invalido] = np.nan
    dataset[col_yoy] = yoy_raw
    cols_yoy.append(col_yoy)

# Feature de aceleração (segunda derivada da margem EBITDA)
if 'margem_ebitda_yoy' in dataset.columns:
    accel = dataset.groupby('CNPJ_CIA')['margem_ebitda_yoy'].diff()
    accel[gap_invalido] = np.nan
    dataset['aceleracao_ebitda'] = accel
    cols_yoy.append('aceleracao_ebitda')

# Feature de posição no ciclo anual (Q1=1, Q2=2, Q3=3, DFP=4)
# Ajuda o modelo a distinguir ITRs acumulados de diferentes momentos do ano
dataset['pos_ciclo'] = dataset['TRIMESTRE'].astype(float)
# Para o DFP de empresas com fiscal year em março (Raízen), trimestre=1 mas é o DFP
# Corrigir: DFP recebe posição 4 independente do trimestre
dataset.loc[dataset['ORIGEM'] == 'DFP', 'pos_ciclo'] = 4.0
cols_yoy.append('pos_ciclo')

# Limpar colunas auxiliares
dataset = dataset.drop(columns=['_dt_prev', '_gap_dias'])

logger.info("YoY: %d features criadas | nulos médios: %.0f%%",
            len(cols_yoy), dataset[cols_yoy].isnull().mean().mean() * 100)
print(f"Features YoY: {len(cols_yoy)} | nulos médios: {dataset[cols_yoy].isnull().mean().mean():.0%}")
print(f"Dataset: {dataset.shape}")


2026-04-22 12:11:20 | INFO     | YoY: gaps inválidos anulados: 107 / 471 obs (22.7%)
2026-04-22 12:11:20 | INFO     | YoY: 18 features criadas | nulos médios: 22%


Features YoY: 18 | nulos médios: 22%
Dataset: (471, 465)


## Etapa 7 — Codificação one-hot do setor e flag de origem

Além do setor, adiciona uma flag binária `flag_dfp` que indica se a observação
é um DFP anual (1.0) ou ITR trimestral (0.0). Isso permite que o modelo aprenda
que a mesma empresa reporta valores acumulados diferentes dependendo do tipo de documento.

In [8]:
# ── One-hot do setor ─────────────────────────────────────────────────────
dataset = pd.get_dummies(dataset, columns=['SETOR'], prefix='setor', dtype=float)
cols_setor = sorted([c for c in dataset.columns if c.startswith('setor_')])

# ── Flag de origem ────────────────────────────────────────────────────────
# O modelo precisa saber se está vendo um período parcial (ITR) ou completo (DFP)
# Um ITR de setembro reporta 75% do resultado anual — valores intrinsecamente menores
dataset['flag_dfp'] = (dataset['ORIGEM'] == 'DFP').astype(float)

logger.info("One-hot SETOR: %d colunas | flag_dfp criada", len(cols_setor))
print(f"Setores: {cols_setor}")
print(f"flag_dfp: {dataset['flag_dfp'].value_counts().to_dict()}")
print(f"Dataset: {dataset.shape}")


2026-04-22 12:11:20 | INFO     | One-hot SETOR: 5 colunas | flag_dfp criada


Setores: ['setor_Commodities', 'setor_Energia', 'setor_Petróleo', 'setor_Tecnologia', 'setor_Varejo']
flag_dfp: {0.0: 372, 1.0: 99}
Dataset: (471, 470)


## Etapa 8 — Criação dos targets: próximo DFP anual

**Regra fundamental:** cada observação (DFP ou ITR) prediz o valor do
**primeiro DFP estritamente posterior** à sua data de referência.

Isso é implementado por lookup direto — para cada linha da empresa E na data D,
busca o menor `DT_REFER` do DFP de E com `DT_REFER > D`.

Consequências:
- ITR Q1/2022 → target = DFP 2022 (próximo DFP após março)
- ITR Q3/2022 → target = DFP 2022 (mesmo target, mas com mais informação acumulada)
- DFP 2022    → target = DFP 2023 (o DFP prediz o *próximo* DFP, não a si mesmo)
- ITRs do último ano disponível → sem target (DFP futuro ainda não existe)

In [9]:
# ── Tabela de targets: somente os DFPs ───────────────────────────────────
dfp_targets = (
    dataset[dataset['ORIGEM'] == 'DFP']
    [['CNPJ_CIA', 'DT_REFER'] + list(TARGET_COLS.values())]
    .rename(columns={'DT_REFER': 'DT_DFP',
                     **{v: k for k, v in TARGET_COLS.items()}})
    .sort_values(['CNPJ_CIA', 'DT_DFP'])
    .reset_index(drop=True)
)

# ── Lookup: próximo DFP estritamente posterior por empresa ────────────────
# Implementação por empresa (evita vazamento entre CNPJs no merge_asof)
partes = []
for cnpj, grupo in dataset.sort_values(['CNPJ_CIA', 'DT_REFER']).groupby('CNPJ_CIA'):
    dfp_emp = (dfp_targets[dfp_targets['CNPJ_CIA'] == cnpj]
               .sort_values('DT_DFP')
               .reset_index(drop=True))
    g = grupo.sort_values('DT_REFER').copy()

    for idx in g.index:
        dt_linha = g.loc[idx, 'DT_REFER']
        # Próximo DFP ESTRITAMENTE posterior (DT_DFP > DT_REFER, não >=)
        proximos = dfp_emp[dfp_emp['DT_DFP'] > dt_linha]
        if len(proximos):
            proximo = proximos.iloc[0]
            for tgt_col in TARGET_COLS:
                g.loc[idx, tgt_col]   = proximo[tgt_col]
            g.loc[idx, 'DT_TARGET'] = proximo['DT_DFP']
    partes.append(g)

dataset = pd.concat(partes, ignore_index=True)
targets_criados = [t for t in TARGET_COLS if t in dataset.columns]

# ── Diagnóstico ───────────────────────────────────────────────────────────
for tgt in targets_criados:
    n_val = dataset[tgt].notna().sum()
    logger.info("Target %-22s: %d/%d obs válidas (%.0f%%)",
                tgt, n_val, len(dataset), n_val/len(dataset)*100)

print("\nTargets criados:")
for orig in ['DFP', 'ITR']:
    sub = dataset[dataset['ORIGEM'] == orig]
    val = sub[targets_criados[0]].notna().sum()
    print(f"  {orig}: {val}/{len(sub)} obs com target válido")

# Remove observações sem nenhum target (últimos períodos de cada empresa)
n_ant = len(dataset)
dataset = dataset[dataset[targets_criados].notna().any(axis=1)].reset_index(drop=True)
logger.info("Remoção sem target: %d → %d (-%d)", n_ant, len(dataset), n_ant - len(dataset))
print(f"\nDataset com targets: {dataset.shape[0]} × {dataset.shape[1]}")


2026-04-22 12:11:20 | INFO     | Target TARGET_DRE_3.01       : 369/471 obs válidas (78%)
2026-04-22 12:11:20 | INFO     | Target TARGET_DRE_3.11       : 369/471 obs válidas (78%)
2026-04-22 12:11:20 | INFO     | Target TARGET_EBITDA         : 369/471 obs válidas (78%)
2026-04-22 12:11:20 | INFO     | Remoção sem target: 471 → 369 (-102)



Targets criados:
  DFP: 74/99 obs com target válido
  ITR: 295/372 obs com target válido

Dataset com targets: 369 × 474


## Etapa 9 — Seleção de features por correlação e RFE

Pool de features candidatas inclui KPIs, YoY, setor (one-hot), flag_dfp e pos_ciclo.
A seleção usa correlação de Pearson como pré-filtro e RFE com Ridge como seletor final.

`GroupKFold` por empresa é usado na validação interna do RFE para evitar que
observações da mesma empresa contaminem treino e validação.

In [10]:
FEATURES_CANDIDATAS = (
    kpis_presentes
    + cols_yoy
    + cols_setor
    + ['flag_dfp', 'pos_ciclo']
    + (['aceleracao_ebitda'] if 'aceleracao_ebitda' in dataset.columns else [])
)
FEATURES_CANDIDATAS = [f for f in FEATURES_CANDIDATAS if f in dataset.columns]
logger.info("Features candidatas: %d", len(FEATURES_CANDIDATAS))

target_principal = 'TARGET_DRE_3.01'

if target_principal not in dataset.columns:
    logger.error("Target principal não encontrado")
    FEATURES_SELECIONADAS = FEATURES_CANDIDATAS[:N_FEATURES_RFE]
else:
    df_sel = dataset[FEATURES_CANDIDATAS + [target_principal]].dropna()
    X_sel  = df_sel[FEATURES_CANDIDATAS]
    y_sel  = df_sel[target_principal]

    # ── Pré-seleção por correlação ────────────────────────────────────────
    corr_abs = X_sel.corrwith(y_sel).abs().sort_values(ascending=False)
    FEATURES_CORR = corr_abs[corr_abs >= CORR_MIN].index.tolist()
    logger.info("Pearson (|r| >= %.2f): %d → %d features",
                CORR_MIN, len(FEATURES_CANDIDATAS), len(FEATURES_CORR))

    # ── RFE com Ridge ─────────────────────────────────────────────────────
    n_rfe = min(N_FEATURES_RFE, len(FEATURES_CORR))
    if len(FEATURES_CORR) <= n_rfe:
        FEATURES_SELECIONADAS = FEATURES_CORR
    else:
        scaler   = StandardScaler()
        X_scaled = scaler.fit_transform(df_sel[FEATURES_CORR])
        rfe = RFE(Ridge(alpha=1.0), n_features_to_select=n_rfe, step=2)
        rfe.fit(X_scaled, y_sel)
        FEATURES_SELECIONADAS = [FEATURES_CORR[i]
                                 for i, sel in enumerate(rfe.support_) if sel]
        logger.info("RFE: %d → %d features", len(FEATURES_CORR), len(FEATURES_SELECIONADAS))

    print(f"\nFeatures selecionadas ({len(FEATURES_SELECIONADAS)}):")
    for f in FEATURES_SELECIONADAS:
        print(f"  {f:<35} |r| = {corr_abs.get(f, 0):.3f}")


2026-04-22 12:11:20 | INFO     | Features candidatas: 45
2026-04-22 12:11:20 | INFO     | Pearson (|r| >= 0.10): 45 → 20 features
2026-04-22 12:11:21 | INFO     | RFE: 20 → 15 features



Features selecionadas (15):
  div_liquida                         |r| = 0.830
  EBITDA                              |r| = 0.709
  FCF                                 |r| = 0.676
  setor_Petróleo                      |r| = 0.644
  liquidez_corrente                   |r| = 0.358
  setor_Tecnologia                    |r| = 0.255
  giro_ativo                          |r| = 0.238
  margem_bruta                        |r| = 0.228
  setor_Energia                       |r| = 0.225
  setor_Varejo                        |r| = 0.216
  roa                                 |r| = 0.167
  roe                                 |r| = 0.164
  fco_receita_yoy                     |r| = 0.138
  endividamento_yoy                   |r| = 0.108
  alavancagem_de_yoy                  |r| = 0.102


## Etapa 10 — Split temporal treino/teste

O split respeita a ordem cronológica dentro de cada empresa. Os 75% de períodos
mais antigos vão para treino, os 25% mais recentes para teste.

Com DFP + ITR, cada empresa contribui com ~15 períodos (3 ITRs + 1 DFP por ano
× ~4 anos com target). O teste captura os períodos mais recentes — exatamente
o cenário real de predição onde o modelo usa dados históricos para prever o futuro.

In [11]:
def split_temporal_empresa(grupo: pd.DataFrame, frac: float = 0.75) -> pd.DataFrame:
    """
    Marca os `frac` períodos mais antigos como treino e o restante como teste.
    Garante pelo menos 1 obs em cada split.
    """
    grupo = grupo.sort_values('DT_REFER').copy()
    n = len(grupo)
    n_tr = max(1, round(n * frac))
    n_tr = min(n_tr, n - 1)
    grupo['split'] = 'teste'
    grupo.iloc[:n_tr, grupo.columns.get_loc('split')] = 'treino'
    return grupo

dataset = dataset.groupby('CNPJ_CIA', group_keys=False).apply(
    split_temporal_empresa, frac=FRAC_TREINO
)

treino = dataset[dataset['split'] == 'treino'].copy()
teste  = dataset[dataset['split'] == 'teste'].copy()

# Verificação anti-contaminação: max(ano treino) < min(ano teste) por empresa
for cnpj, grp in dataset.groupby('CNPJ_CIA'):
    anos_tr = set(grp[grp['split']=='treino']['ANO'].dropna().astype(int))
    anos_te = set(grp[grp['split']=='teste']['ANO'].dropna().astype(int))
    if anos_tr and anos_te and max(anos_tr) > min(anos_te):
        logger.warning("Contaminação temporal | empresa %s | "
                       "treino max=%d > teste min=%d",
                       cnpj, max(anos_tr), min(anos_te))

n_tot = len(dataset)
logger.info("Split: %d treino (%.0f%%) | %d teste (%.0f%%)",
            len(treino), len(treino)/n_tot*100,
            len(teste),  len(teste)/n_tot*100)

print(f"Treino: {len(treino)} obs ({len(treino)/n_tot:.0%})")
print(f"Teste : {len(teste)} obs ({len(teste)/n_tot:.0%})")
print()
print("Distribuição por empresa (ORIGEM × split):")
dist = (dataset.groupby(['NOME_CIA', 'ORIGEM', 'split'])
        .size().unstack(['ORIGEM','split'], fill_value=0))
print(dist.to_string())


KeyError: 'CNPJ_CIA'

## Etapa 11 — Persistência dos artefatos

Salva todos os artefatos para o Script 3, incluindo um `grupos_treino` que mapeia
cada observação de treino à sua empresa — necessário para o `GroupKFold` no Script 3.

In [ ]:
TARGETS_VALIDOS = [t for t in TARGET_COLS if t in dataset.columns]

# grupos_treino: array de CNPJ para uso com GroupKFold no Script 3
# GroupKFold garante que todas as obs de uma empresa ficam no mesmo fold
grupos_treino = treino['CNPJ_CIA'].values

artefatos_df = {
    'dataset_preparado': dataset,
    'treino'           : treino,
    'teste'            : teste,
}
artefatos_meta = {
    'features'      : FEATURES_SELECIONADAS,
    'targets'       : TARGETS_VALIDOS,
    'kpis'          : kpis_presentes,
    'cols_setor'    : cols_setor,
    'cols_yoy'      : cols_yoy,
    'grupos_treino' : grupos_treino,   # ← novo: para GroupKFold no Script 3
    'params': {
        'versao'         : 'V3_DFP_ITR',
        'limiar_nulo'    : LIMIAR_NULO,
        'fator_winsor'   : FATOR_WINSOR,
        'max_cols_yoy'   : MAX_COLS_YOY,
        'clip_yoy'       : CLIP_YOY,
        'corr_min'       : CORR_MIN,
        'n_features_rfe' : N_FEATURES_RFE,
        'frac_treino'    : FRAC_TREINO,
        'gap_yoy_min'    : GAP_YOY_MIN,
        'gap_yoy_max'    : GAP_YOY_MAX,
        'estrategia_target': 'proximo_dfp_estritamente_posterior',
    },
}

for nome, df_art in artefatos_df.items():
    cam = PASTA_SAIDA / f'{nome}.parquet'
    df_art.to_parquet(cam, index=False)
    logger.info("Salvo: %s | %d × %d | %.0f KB",
                cam.name, *df_art.shape, cam.stat().st_size / 1024)

for nome, obj in artefatos_meta.items():
    cam = PASTA_SAIDA / f'{nome}.pkl'
    with open(cam, 'wb') as f:
        pickle.dump(obj, f)
    logger.info("Salvo: %s", cam.name)

relatorio = {
    'versao'                  : 'V3_DFP_ITR',
    'estrategia_target'       : 'proximo_dfp_estritamente_posterior',
    'dataset_empresas'        : int(dataset['CNPJ_CIA'].nunique()),
    'dataset_anos'            : sorted(dataset['ANO'].dropna().astype(int).unique().tolist()),
    'n_obs_dfp'               : int((dataset['ORIGEM']=='DFP').sum()),
    'n_obs_itr'               : int((dataset['ORIGEM']=='ITR').sum()),
    'n_obs_total'             : int(len(dataset)),
    'n_obs_treino'            : int(len(treino)),
    'n_obs_treino_dfp'        : int((treino['ORIGEM']=='DFP').sum()),
    'n_obs_treino_itr'        : int((treino['ORIGEM']=='ITR').sum()),
    'n_obs_teste'             : int(len(teste)),
    'n_features_candidatas'   : int(len(FEATURES_CANDIDATAS)),
    'n_features_selecionadas' : int(len(FEATURES_SELECIONADAS)),
    'targets'                 : TARGETS_VALIDOS,
    'features'                : FEATURES_SELECIONADAS,
    'params'                  : artefatos_meta['params'],
    'nota_autocorrelacao'     : (
        'Usar GroupKFold(groups=grupos_treino) no Script 3 para '
        'evitar que obs da mesma empresa contaminem treino/validação no CV.'
    ),
}
cam_rel = PASTA_SAIDA / 'relatorio_preparacao_v3.json'
with open(cam_rel, 'w', encoding='utf-8') as f:
    json.dump(relatorio, f, indent=2, ensure_ascii=False, default=str)

# ── Resumo final ──────────────────────────────────────────────────────────
print("\n" + "═"*65)
print("  RESUMO FINAL — Script 2 V3 (DFP + ITR)")
print("═"*65)
print(f"  Input         : {cam_parquet.name}")
print(f"  Total obs     : {len(dataset)} (DFP: {(dataset['ORIGEM']=='DFP').sum()} | ITR: {(dataset['ORIGEM']=='ITR').sum()})")
print(f"  Empresas      : {dataset['CNPJ_CIA'].nunique()} / 25")
print(f"  Anos          : {sorted(dataset['ANO'].dropna().astype(int).unique())}")
print(f"  KPIs          : {len(kpis_presentes)} / {len(LISTA_KPIS)}")
print(f"  Features      : {len(FEATURES_SELECIONADAS)} selecionadas")
print(f"  Targets       : {TARGETS_VALIDOS}")
print(f"  Treino        : {len(treino)} obs ({len(treino)/len(dataset):.0%})")
print(f"    ↳ DFP       : {(treino['ORIGEM']=='DFP').sum()}")
print(f"    ↳ ITR       : {(treino['ORIGEM']=='ITR').sum()}")
print(f"  Teste         : {len(teste)} obs ({len(teste)/len(dataset):.0%})")
print(f"  Vs. V2 (DFP): {len(dataset)/74:.1f}× mais observações")
print(f"  Artefatos     : {PASTA_SAIDA}/")
print("═"*65)
print("  ⚠️  Script 3: usar GroupKFold(groups=grupos_treino) no CV")
print("  ✅  Pronto para o Script 3 (03_cvm_modelagem.ipynb)")
print("═"*65)
